# Bank Marketing – Data Preprocessing Pipeline
Target: `y` (did the client subscribe to a term deposit? yes / no).

**Design rules followed throughout**
* Exact-duplicate removal and the deterministic `pdays` fix are applied *before* the split (they learn nothing from the data).
* The 80:20 stratified split happens **before** any decision that uses data statistics.
* Outlier fences, clipping limits, encoder categories, scaler mean/std and SMOTE are all fitted on the **training set only**.
* The test set is only *transformed* (never fitted on, never resampled, never plotted for decisions).

## 0. Imports and configuration

In [ ]:
import os
import sys
import json
import subprocess
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

# ---------------- Configuration (change here) ----------------
RANDOM_STATE = 42
TEST_SIZE = 0.20                    # 80:20 split
INCLUDE_DURATION = False            # False = leakage-safe (see Step 8)
WINSORIZE_OUTLIERS = True           # clip extreme values (records are NEVER removed)
CLIP_QUANTILES = (0.01, 0.99)       # limits learned from the training set only
APPLY_SMOTE = True                  # build a SMOTE-resampled TRAIN set (test untouched)
OUT_DIR = os.environ.get("OUT_DIR", "processed_data")
PLOT_DIR = os.path.join(OUT_DIR, "plots")
os.makedirs(PLOT_DIR, exist_ok=True)

TARGET = "y"                        # original label column (yes/no)
TARGET_NUM = "y_encoded"            # numeric target (yes=1, no=0)
LABEL_MAP = {"no": 0, "yes": 1}
INV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}


def find_data_file():
    """Locate bank-full.csv locally, in Colab, or ask for an upload in Colab."""
    candidates = [
        os.environ.get("DATA_PATH", ""),
        "bank-full.csv",
        "../data/raw/bank-full.csv",
        "data/raw/bank-full.csv",
        "/content/bank-full.csv",
        "/mnt/user-data/uploads/bank-full.csv",
    ]
    for p in candidates:
        if p and os.path.exists(p):
            return p
    try:  # Google Colab upload dialog
        from google.colab import files
        print("Upload bank-full.csv ...")
        return list(files.upload().keys())[0]
    except ImportError:
        raise FileNotFoundError("bank-full.csv not found. Put it next to this notebook.")


DATA_PATH = find_data_file()
print("Using data file:", DATA_PATH)

## 1. Load data
The file is semicolon-separated, so `sep=';'` is required.

In [ ]:
df_raw = pd.read_csv(DATA_PATH, sep=";")
print("Original shape:", df_raw.shape)
df_raw.head()

## Step 4 – Duplicates
Exact duplicates (all 17 columns identical) carry no extra information and can bias
training/testing if copies land in both sets, so they are removed *before* splitting.
The dataset has no customer ID, so row-level equality is the only possible test.

In [ ]:
n_duplicates = int(df_raw.duplicated().sum())
print("Duplicate rows found:", n_duplicates)
df = df_raw.drop_duplicates().reset_index(drop=True)
print("Shape after removing duplicates:", df.shape)

### Missing-value audit
Two kinds of "missing" are checked:
1. true `NaN`/blank cells,
2. *hidden* missing values: the string `"unknown"` in categorical columns and the sentinel `pdays == -1` ("never contacted before").

This is a descriptive audit of the raw file. Any modelling decision based on it
(imputation values, encodings) is learned from the training set later.

In [ ]:
cat_like = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
missing_summary = pd.DataFrame({
    "n_nan": df.isna().sum(),
    "n_unknown": [(df[c] == "unknown").sum() if c in cat_like else 0 for c in df.columns],
    "n_pdays_sentinel": [(df[c] == -1).sum() if c == "pdays" else 0 for c in df.columns],
})
missing_summary["n_hidden_or_true_missing"] = missing_summary.sum(axis=1)
missing_summary["pct"] = (missing_summary["n_hidden_or_true_missing"] / len(df) * 100).round(2)
print(missing_summary[missing_summary["n_hidden_or_true_missing"] > 0]
      .sort_values("pct", ascending=False))
print("\nTotal true NaN cells:", int(df.isna().sum().sum()))

## Step 7 – Encode the target
`yes -> 1`, `no -> 0` in a **new** column; the original `y` labels are kept for reporting.

In [ ]:
df[TARGET_NUM] = df[TARGET].map(LABEL_MAP)
assert df[TARGET_NUM].notna().all(), "Unexpected label in target column"
print(df[[TARGET, TARGET_NUM]].value_counts().sort_index())

## Deterministic feature fix: `pdays`
`pdays = -1` is a placeholder meaning "client was not contacted in a previous campaign",
not a real number of days. Leaving -1 in would distort scaling and outlier detection.
It is converted into an explicit 0/1 flag `previously_contacted`, and `pdays` is set to 0
for those clients. This rule uses no statistics from the data, so applying it before the
split cannot leak. The same function is reused for future/new customers.

In [ ]:
def add_engineered_features(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out["previously_contacted"] = (out["pdays"] != -1).astype(int)
    out["pdays"] = out["pdays"].where(out["pdays"] != -1, 0)
    return out


df = add_engineered_features(df)
print(df[["pdays", "previously_contacted"]].describe().T)

## Step 8 – Feature selection (X vs y)
* **Identifiers:** the code below checks for ID-like columns (name or all-unique values). None is expected here.
* **`duration` (call length in seconds) – excluded by default.**
  It is only known *after* the call ends, and a very short call almost always means "no".
  If the model is meant to score customers **before** a call is made, including `duration`
  would leak the outcome and give unrealistically high metrics. Keep it only for
  explanatory/benchmark analysis (set `INCLUDE_DURATION = True`) and report that it is not deployable.
  Evidence from the training data is printed after the split (Step 9).

In [ ]:
id_like = [c for c in df.columns
           if c.lower() in {"id", "customer_id", "customerid", "index"}
           or (df[c].nunique() == len(df) and c not in (TARGET, TARGET_NUM))]
print("Identifier-like columns:", id_like if id_like else "none found")

drop_cols = [TARGET, TARGET_NUM] + id_like
if not INCLUDE_DURATION:
    drop_cols.append("duration")

feature_cols = [c for c in df.columns if c not in drop_cols]
print("\nSelected features (%d):" % len(feature_cols), feature_cols)
print("Excluded:", drop_cols)

## Step 9 – Stratified 80:20 train/test split
Done on the whole frame (so raw train/test files can also be saved). From here on the
test set is not inspected for any decision.

In [ ]:
train_df, test_df = train_test_split(
    df, test_size=TEST_SIZE, stratify=df[TARGET_NUM], random_state=RANDOM_STATE
)
X_train, X_test = train_df[feature_cols].copy(), test_df[feature_cols].copy()
y_train, y_test = train_df[TARGET_NUM].copy(), test_df[TARGET_NUM].copy()

assert set(X_train.index).isdisjoint(set(X_test.index)), "Train/test overlap!"
print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("Share of 'yes' -> full: %.4f | train: %.4f | test: %.4f"
      % (df[TARGET_NUM].mean(), y_train.mean(), y_test.mean()))

# Evidence for the `duration` decision, computed on TRAIN only
dur_corr = train_df["duration"].corr(train_df[TARGET_NUM])
dur_by_class = train_df.groupby(TARGET)["duration"].agg(["mean", "median"]).round(1)
dur_bins = (train_df.groupby(pd.cut(train_df["duration"], [-1, 100, 300, 600, 1000, np.inf]))
            [TARGET_NUM].mean().round(3))
print("\n[Train only] correlation(duration, y):", round(dur_corr, 3))
print(dur_by_class)
print("Share of 'yes' by call-duration bin:\n", dur_bins)

## Step 6 – Outlier detection (IQR method, training set only)
Fences are `Q1 - 1.5*IQR` and `Q3 + 1.5*IQR`. This is **detection and reporting**: no customer
record is removed. Large `balance`, `campaign` or `age` values are valid customers.
Note that `previous`/`pdays` are flagged mostly because ~82% of clients were never contacted
before (zeros), which is structural rather than erroneous.

In [ ]:
eda_num_cols = [c for c in train_df.columns
                if pd.api.types.is_numeric_dtype(train_df[c])
                and c not in (TARGET_NUM, "previously_contacted")]


def iqr_outlier_table(frame: pd.DataFrame, cols) -> pd.DataFrame:
    rows = []
    for c in cols:
        q1, q3 = frame[c].quantile([0.25, 0.75])
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        n_lo, n_hi = int((frame[c] < lo).sum()), int((frame[c] > hi).sum())
        rows.append({"feature": c, "lower_fence": lo, "upper_fence": hi,
                     "n_below": n_lo, "n_above": n_hi,
                     "pct_outliers": round((n_lo + n_hi) / len(frame) * 100, 2),
                     "min": frame[c].min(), "max": frame[c].max()})
    return pd.DataFrame(rows).set_index("feature")


outlier_table = iqr_outlier_table(train_df, eda_num_cols)
print(outlier_table)

## Steps 5, 6, 10 – Encoding, scaling and the leakage-safe pipeline
| Column group | Treatment |
|---|---|
| Binary yes/no (`default`, `housing`, `loan`) | mapped `no=0, yes=1` (OrdinalEncoder with fixed order) |
| Flag (`previously_contacted`) | already 0/1, passed through |
| Multi-category (`job`, `marital`, `education`, `contact`, `month`, `poutcome`) | One-Hot Encoding; `"unknown"` kept as its own category; unseen categories ignored |
| Numeric | median imputer (safety net) -> winsorise at train 1%/99% -> StandardScaler |

Scaling matters for distance/gradient models (Logistic Regression, KNN, SVM, neural nets);
tree ensembles do not need it but are not harmed by it.
Everything is fitted inside a `ColumnTransformer` on `X_train` only.

In [ ]:
class QuantileClipper(BaseEstimator, TransformerMixin):
    """Winsorise each column to quantile limits learned in fit() (train only)."""

    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.lower_ = np.quantile(X, self.lower, axis=0)
        self.upper_ = np.quantile(X, self.upper, axis=0)
        return self

    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.lower_, self.upper_)

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)


flag_cols = ["previously_contacted"]
non_numeric = [c for c in feature_cols if not pd.api.types.is_numeric_dtype(X_train[c])]
binary_cols = [c for c in non_numeric if set(X_train[c].unique()) <= {"yes", "no"}]
cat_cols = [c for c in non_numeric if c not in binary_cols]
num_cols = [c for c in feature_cols
            if c not in non_numeric and c not in flag_cols]

assert sorted(binary_cols + cat_cols + num_cols + flag_cols) == sorted(feature_cols)
print("Binary     :", binary_cols)
print("Flag       :", flag_cols)
print("Categorical:", cat_cols)
print("Numeric    :", num_cols)

num_steps = [("impute", SimpleImputer(strategy="median"))]
if WINSORIZE_OUTLIERS:
    num_steps.append(("clip", QuantileClipper(*CLIP_QUANTILES)))
num_steps.append(("scale", StandardScaler()))

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline(num_steps), num_cols),
        ("bin", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("encode", OrdinalEncoder(categories=[["no", "yes"]] * len(binary_cols),
                                      handle_unknown="use_encoded_value",
                                      unknown_value=-1)),
        ]), binary_cols),
        ("flag", "passthrough", flag_cols),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value="unknown")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), cat_cols),
    ],
    verbose_feature_names_out=False,
)

# FIT on train only, TRANSFORM both
preprocessor.fit(X_train)
feature_names = list(preprocessor.get_feature_names_out())
X_train_p = pd.DataFrame(preprocessor.transform(X_train), columns=feature_names, index=X_train.index)
X_test_p = pd.DataFrame(preprocessor.transform(X_test), columns=feature_names, index=X_test.index)

# Verify the scaler really saw only training rows
n_seen = preprocessor.named_transformers_["num"].named_steps["scale"].n_samples_seen_
assert n_seen == len(X_train), "Scaler was fitted on more than the training data!"
assert not X_train_p.isna().any().any() and not X_test_p.isna().any().any()
print(f"\nScaler fitted on {int(n_seen)} rows (= training rows only).")
print("Processed shapes -> train:", X_train_p.shape, "| test:", X_test_p.shape)
X_train_p.head()

## Step 11 – Class imbalance
Accuracy is misleading on imbalanced data (always predicting "no" scores ~88%). Options:
* **Class weights** (`class_weight='balanced'`, or `scale_pos_weight` in XGBoost): no new rows, simplest and safest default.
* **SMOTE**: creates synthetic minority samples by interpolating neighbours. Apply **only to the training data**, and inside cross-validation folds
  (use `imblearn.pipeline.Pipeline`) when tuning, otherwise validation folds leak.
  On one-hot data SMOTE yields fractional dummy values; `SMOTENC` on the pre-encoded frame is an alternative.
* **Threshold tuning / PR-AUC / F1 / recall** for evaluation, regardless of the method used.

Below: class weights are computed, and a SMOTE-resampled copy of the *training* set is created. The test set is untouched.

In [ ]:
class_counts = y_train.value_counts().sort_index()
imbalance_ratio = class_counts[0] / class_counts[1]
print("Training class counts:\n", class_counts.rename(INV_LABEL_MAP))
print("Imbalance ratio (no : yes) = %.2f : 1" % imbalance_ratio)

weights = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_train)
class_weight_dict = {0: float(weights[0]), 1: float(weights[1])}
print("Balanced class weights (use as class_weight=...):", class_weight_dict)

X_train_smote, y_train_smote = None, None
if APPLY_SMOTE:
    try:
        try:
            from imblearn.over_sampling import SMOTE
        except ImportError:  # Colab usually has it; otherwise install once
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "imbalanced-learn"])
            from imblearn.over_sampling import SMOTE
        smote = SMOTE(random_state=RANDOM_STATE)
        Xs, ys = smote.fit_resample(X_train_p, y_train)          # TRAIN ONLY
        X_train_smote = pd.DataFrame(Xs, columns=feature_names)
        y_train_smote = pd.Series(ys, name=TARGET_NUM)
        print("\nAfter SMOTE (train only):\n",
              y_train_smote.value_counts().sort_index().rename(INV_LABEL_MAP))
        print("Test set unchanged:", X_test_p.shape, "| class counts:",
              y_test.value_counts().sort_index().to_dict())
    except Exception as e:  # keep the rest of the notebook working
        print("SMOTE skipped (install `imbalanced-learn` to enable):", e)

## Step 12 – Save processed data and preprocessing objects

In [ ]:
# Cleaned full dataset (duplicates removed, original label + numeric target kept)
df.to_csv(os.path.join(OUT_DIR, "bank_cleaned.csv"), index=False)

# Raw (un-encoded) split – handy for sklearn Pipelines / cross-validation
train_df.to_csv(os.path.join(OUT_DIR, "train_raw.csv"), index=False)
test_df.to_csv(os.path.join(OUT_DIR, "test_raw.csv"), index=False)

# Fully processed split
X_train_p.to_csv(os.path.join(OUT_DIR, "X_train_processed.csv"), index=False)
X_test_p.to_csv(os.path.join(OUT_DIR, "X_test_processed.csv"), index=False)
y_train.to_csv(os.path.join(OUT_DIR, "y_train.csv"), index=False)
y_test.to_csv(os.path.join(OUT_DIR, "y_test.csv"), index=False)

if X_train_smote is not None:
    X_train_smote.to_csv(os.path.join(OUT_DIR, "X_train_smote.csv"), index=False)
    y_train_smote.to_csv(os.path.join(OUT_DIR, "y_train_smote.csv"), index=False)

# Preprocessing objects for future predictions
joblib.dump(preprocessor, os.path.join(OUT_DIR, "preprocessor.joblib"))
with open(os.path.join(OUT_DIR, "preprocessing_config.json"), "w") as f:
    json.dump({
        "feature_cols_raw": feature_cols,
        "feature_names_processed": feature_names,
        "binary_cols": binary_cols, "flag_cols": flag_cols,
        "cat_cols": cat_cols, "num_cols": num_cols,
        "include_duration": INCLUDE_DURATION,
        "label_map": LABEL_MAP,
        "class_weights": class_weight_dict,
        "random_state": RANDOM_STATE, "test_size": TEST_SIZE,
    }, f, indent=2)

print("Saved files:", sorted(os.listdir(OUT_DIR)))
print("""
To score new customers later:
    new = add_engineered_features(new_raw_df)[feature_cols]
    X_new = preprocessor.transform(new)
""")

## Step 13 – Visualisations
Plots that describe data used for decisions (distributions, correlations) use the **training set only**.
The target-distribution and missing-value plots are descriptive audits of the whole file.

In [ ]:
# 13a. Target distribution (whole cleaned dataset)
fig, ax = plt.subplots(figsize=(5, 4))
order = ["no", "yes"]
sns.countplot(x=TARGET, data=df, order=order, palette=["#4C72B0", "#DD8452"], ax=ax)
for i, lab in enumerate(order):
    n = int((df[TARGET] == lab).sum())
    ax.text(i, n, f"{n:,}\n({n / len(df) * 100:.1f}%)", ha="center", va="bottom")
ax.set_title("Target distribution (y)")
ax.set_ylim(0, df[TARGET].value_counts().max() * 1.18)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "01_target_distribution.png"), dpi=150)
plt.show()

In [ ]:
# 13b. Missing-values summary (true NaN + 'unknown' + pdays sentinel)
ms = missing_summary[missing_summary["n_hidden_or_true_missing"] > 0].sort_values("pct")
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(ms.index, ms["pct"], color="#C44E52")
for y_pos, v in enumerate(ms["pct"]):
    ax.text(v + 0.5, y_pos, f"{v:.1f}%", va="center")
ax.set_xlabel("% of rows")
ax.set_title(f"Hidden missing values ('unknown' / pdays=-1)   |   true NaN cells: {int(df_raw.isna().sum().sum())}")
ax.set_xlim(0, max(ms["pct"].max() * 1.15, 1))
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "02_missing_values.png"), dpi=150)
plt.show()

In [ ]:
# 13c. Numerical feature distributions (train, raw scale) + boxplots showing outliers
n = len(eda_num_cols)
fig, axes = plt.subplots(2, n, figsize=(2.9 * n, 6))
for j, c in enumerate(eda_num_cols):
    sns.histplot(train_df[c], bins=40, color="#4C72B0", ax=axes[0, j])
    axes[0, j].set_title(c)
    axes[0, j].set_xlabel("")
    sns.boxplot(y=train_df[c], color="#DD8452", fliersize=1.5, ax=axes[1, j])
    axes[1, j].set_ylabel("")
fig.suptitle("Numerical features – training set (raw scale)", y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "03_numeric_distributions.png"), dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
# 13d. Same numeric features AFTER winsorising + scaling (train)
fig, ax = plt.subplots(figsize=(1.3 * len(num_cols) + 2, 4))
sns.boxplot(data=X_train_p[num_cols], fliersize=1.5, color="#55A868", ax=ax)
ax.set_title("Numerical features after clipping & standard scaling (train)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "04_numeric_after_scaling.png"), dpi=150)
plt.show()

In [ ]:
# 13e. Correlation heatmap – numeric features + numeric target (train only)
corr_cols = eda_num_cols + ["previously_contacted", TARGET_NUM]
corr = train_df[corr_cols].corr()
fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=ax)
ax.set_title("Correlation heatmap (train)")
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "05_correlation_heatmap.png"), dpi=150)
plt.show()

In [ ]:
# 13f. Class distribution before / after resampling
panels = [("Train (original)", y_train), ("Test (untouched)", y_test)]
if y_train_smote is not None:
    panels.insert(1, ("Train (after SMOTE)", y_train_smote))
fig, axes = plt.subplots(1, len(panels), figsize=(4.2 * len(panels), 4), sharey=False)
for ax, (title, ys) in zip(np.atleast_1d(axes), panels):
    vc = ys.value_counts().sort_index()
    ax.bar(["no (0)", "yes (1)"], vc.values, color=["#4C72B0", "#DD8452"])
    for i, v in enumerate(vc.values):
        ax.text(i, v, f"{v:,}\n({v / vc.sum() * 100:.1f}%)", ha="center", va="bottom")
    ax.set_title(title)
    ax.set_ylim(0, vc.max() * 1.2)
plt.suptitle("Class distribution before and after resampling", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "06_class_distribution_resampling.png"), dpi=150, bbox_inches="tight")
plt.show()

## Step 14 – Preprocessing report
Every value is computed from the data.

In [ ]:
def dist(ys):
    vc = ys.value_counts().sort_index()
    return {INV_LABEL_MAP[k]: f"{v:,} ({v / len(ys) * 100:.2f}%)" for k, v in vc.items()}


print("=" * 72)
print("PREPROCESSING REPORT")
print("=" * 72)
print(f"Original shape                : {df_raw.shape}")
print(f"Duplicate rows found/removed  : {n_duplicates}")
print(f"Shape after de-duplication    : {df.shape[0]} rows (+ engineered/target columns -> {df.shape[1]} cols)")
print(f"True NaN cells                : {int(df_raw.isna().sum().sum())}")
print("\nHidden missing values ('unknown' / pdays=-1):")
print(missing_summary[missing_summary['n_hidden_or_true_missing'] > 0]
      [['n_unknown', 'n_pdays_sentinel', 'pct']].sort_values('pct', ascending=False).to_string())
print(f"\nTarget encoding               : {LABEL_MAP}")
print(f"Selected raw features ({len(feature_cols)})    : {feature_cols}")
print(f"Excluded columns              : {drop_cols}")
print(f"'duration' included           : {INCLUDE_DURATION}  "
      f"(train corr with y = {dur_corr:.3f}; known only after the call -> leakage risk)")
print(f"\nBinary / flag / one-hot / numeric columns: "
      f"{len(binary_cols)} / {len(flag_cols)} / {len(cat_cols)} / {len(num_cols)}")
print(f"Processed feature count       : {len(feature_names)}")
print("\nIQR outliers in TRAIN (records kept; extremes winsorised at train "
      f"{CLIP_QUANTILES} if enabled):")
print(outlier_table[['lower_fence', 'upper_fence', 'n_below', 'n_above', 'pct_outliers']].to_string())
print("\nClass distribution:")
print("  Full (cleaned) :", dist(df[TARGET_NUM]))
print("  Train          :", dist(y_train))
print("  Test           :", dist(y_test))
if y_train_smote is not None:
    print("  Train + SMOTE  :", dist(y_train_smote))
print(f"  Imbalance ratio (train) = {imbalance_ratio:.2f} : 1 | balanced weights = "
      f"{ {k: round(v, 3) for k, v in class_weight_dict.items()} }")
print(f"\nTrain shape  X: {X_train_p.shape}  y: {y_train.shape}")
print(f"Test  shape  X: {X_test_p.shape}  y: {y_test.shape}")
if X_train_smote is not None:
    print(f"Train+SMOTE  X: {X_train_smote.shape}  y: {y_train_smote.shape}")
print(f"Split ratio (train:test)      : {len(X_train) / len(df):.2f} : {len(X_test) / len(df):.2f}")
print(f"Output folder                 : {os.path.abspath(OUT_DIR)}")
print("=" * 72)